# 04. OCR 강건성 실습: 흐림, 회전, 대비, 문자 오류

OCR 모델 개발은 깨끗한 이미지에서 끝나지 않습니다. 실제 문서는 흐림, 회전, 낮은 대비, 부분 가림, 다국어 문자 때문에 흔들립니다.
이 노트북은 합성 문서 이미지를 만들고, 변형 조건별 오류율을 측정하는 실험 골격을 제공합니다.


In [ ]:
from pathlib import Path
import json
import math
import random
import statistics
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    print("matplotlib을 불러오지 못했습니다:", exc)

ROOT = Path.cwd()
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

random.seed(42)
np.random.seed(42)

def save_json(name, obj):
    path = ARTIFACTS / name
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

def display_df(df, n=20):
    return df.head(n)


In [ ]:
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageOps

def make_doc(text="Invoice No A17\nTotal 42,000 KRW\nStatus PAID", width=520, height=220):
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    font = ImageFont.load_default()
    y = 25
    for line in text.splitlines():
        draw.text((30, y), line, fill="black", font=font)
        y += 38
    return img

def corrupt(img, blur=0, rotate=0, contrast=1.0, noise=0):
    out = img.copy()
    if blur:
        out = out.filter(ImageFilter.GaussianBlur(radius=blur))
    if rotate:
        out = out.rotate(rotate, expand=True, fillcolor="white")
    if contrast != 1.0:
        gray = ImageOps.grayscale(out)
        arr = np.asarray(gray).astype(np.float32)
        arr = np.clip((arr - 128) * contrast + 128, 0, 255).astype(np.uint8)
        out = Image.fromarray(arr).convert("RGB")
    if noise:
        arr = np.asarray(out).astype(np.int16)
        arr = np.clip(arr + np.random.normal(0, noise, arr.shape), 0, 255).astype(np.uint8)
        out = Image.fromarray(arr)
    return out

base_text = "Invoice No A17\nTotal 42,000 KRW\nStatus PAID"
base_img = make_doc(base_text)
variants = {
    "clean": base_img,
    "blur": corrupt(base_img, blur=1.4),
    "rotate": corrupt(base_img, rotate=4),
    "low_contrast": corrupt(base_img, contrast=0.45),
    "noise": corrupt(base_img, noise=18),
}

for name, img in variants.items():
    img.save(ARTIFACTS / f"ocr_variant_{name}.png")

if plt:
    fig, axes = plt.subplots(1, len(variants), figsize=(15, 3))
    for ax, (name, img) in zip(axes, variants.items()):
        ax.imshow(img)
        ax.set_title(name)
        ax.axis("off")
    fig.tight_layout()


In [ ]:
def simulate_ocr(text, condition):
    replacements = {
        "clean": {},
        "blur": {"A17": "AI7", "42,000": "42000"},
        "rotate": {"Invoice": "lnvoice"},
        "low_contrast": {"PAID": "PAlD"},
        "noise": {"Total": "Tota1", "42,000": "42.000"},
    }
    out = text
    for a, b in replacements.get(condition, {}).items():
        out = out.replace(a, b)
    return out

def char_error_rate(ref, hyp):
    a = list(ref.replace("\n", ""))
    b = list(hyp.replace("\n", ""))
    return edit_distance(a, b) / max(1, len(a))

def edit_distance(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        dp[i][0] = i
    for j in range(len(b) + 1):
        dp[0][j] = j
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + (a[i-1] != b[j-1]))
    return dp[-1][-1]

rows = []
for condition in variants:
    hyp = simulate_ocr(base_text, condition)
    rows.append({"condition": condition, "reference": base_text, "ocr_text": hyp, "cer": char_error_rate(base_text, hyp)})

ocr_eval = pd.DataFrame(rows)
ocr_eval.to_csv(ARTIFACTS / "ocr_robustness_eval.csv", index=False, encoding="utf-8-sig")
ocr_eval


## 선택 실습: 실제 OCR 엔진 연결

`pytesseract` 또는 `easyocr`를 설치한 뒤 `variants` 이미지를 실제 OCR에 넣어 보세요.
보고서에는 엔진 제공 회사명/서비스명이 필요한 경우 `xxx OCR API`처럼 마스킹합니다.
